In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from tests.gmail_source_demo import system_prompt

model_name = "Qwen/Qwen3-1.7B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="mps"
)



In [34]:
import requests


def get_weather(city):
    geocode_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        timeout=20,
    )
    geocode_response.raise_for_status()
    geocode_payload = geocode_response.json()
    results = geocode_payload.get("results") or []
    if not results:
        raise ValueError(f"Could not find coordinates for city: {city}")

    location = results[0]
    forecast_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current": "temperature_2m,apparent_temperature,weather_code,wind_speed_10m",
            "timezone": "auto",
        },
        timeout=20,
    )
    forecast_response.raise_for_status()
    current = forecast_response.json()["current"]
    return {
        "city": location["name"],
        "country": location.get("country"),
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "temperature_celsius": current["temperature_2m"],
        "feels_like_celsius": current["apparent_temperature"],
        "weather_code": current["weather_code"],
        "wind_speed_kmh": current["wind_speed_10m"],
        "observed_at": current["time"],
    }


In [ ]:
tool_catlog = [
    {
        "tool_name": "get_weather",
        "arguent": ["city"],
        "description": "this tool fetches live weather for a city via API. it requires city name as argument and returns structured weather data in celsius.",
        "tool_idx": 0,
    }
]

In [12]:
system_prompt = """You are a agent who can fetch weather information. you can use following tools for this"
                 f"tool_catlog = {tool_catlog}

                 output a json having 2 keys 1 = tool name and 2 = argument values
                 """


In [13]:
# prepare the model input
user_prompt = "What is the weather at Kanpur?"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

model_inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,   8315,    879,    646,
           7807,   9104,   1995,     13,    498,    646,    990,   2701,   7375,
            369,    419,    698,    338,    282,      1,  14172,  20825,    839,
            284,    314,  14172,  20825,    839,    630,    338,   2550,    264,
           2951,   3432,    220,     17,   6894,    220,     16,    284,   5392,
            829,    323,    220,     17,    284,   5693,   2750,    198,   1698,
         151645,    198, 151644,    872,    198,   3838,    374,    279,   9104,
            518,  30563,  24998,     30, 151645,    198, 151644,  77091,    198]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='mps:0')}

In [14]:
# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

output_ids

[151667,
 198,
 32313,
 11,
 279,
 1196,
 374,
 10161,
 369,
 279,
 9104,
 304,
 30563,
 24998,
 13,
 6771,
 752,
 1779,
 279,
 7375,
 2500,
 13,
 576,
 5392,
 16403,
 5646,
 264,
 9104,
 5392,
 11,
 714,
 358,
 1184,
 311,
 1281,
 2704,
 432,
 594,
 15614,
 13,
 576,
 1196,
 2578,
 537,
 387,
 7853,
 315,
 279,
 5392,
 594,
 13885,
 11,
 773,
 358,
 1265,
 9934,
 1105,
 311,
 3410,
 279,
 3283,
 829,
 13,
 13824,
 11,
 279,
 1196,
 11689,
 9733,
 30563,
 24998,
 11,
 773,
 358,
 1265,
 990,
 279,
 9104,
 5392,
 448,
 279,
 3283,
 5733,
 738,
 311,
 30563,
 24998,
 13,
 6771,
 752,
 7683,
 279,
 5392,
 594,
 5029,
 13,
 576,
 5392,
 7460,
 264,
 3283,
 829,
 11,
 773,
 358,
 3278,
 5944,
 279,
 4718,
 448,
 330,
 14172,
 1269,
 1,
 438,
 330,
 455,
 69364,
 1,
 323,
 330,
 14479,
 9146,
 1,
 8482,
 330,
 8926,
 788,
 330,
 42,
 276,
 24998,
 3263,
 2938,
 1265,
 7807,
 279,
 9104,
 821,
 12440,
 624,
 151668,
 271,
 515,
 220,
 330,
 14172,
 1269,
 788,
 330,
 455,
 69364,
 756,
 220,


In [15]:
# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


thinking content: <think>
Okay, the user is asking for the weather in Kanpur. Let me check the tools available. The tool catalog includes a weather tool, but I need to make sure it's accessible. The user might not be aware of the tool's existence, so I should prompt them to provide the city name. Wait, the user specifically mentioned Kanpur, so I should use the weather tool with the city parameter set to Kanpur. Let me confirm the tool's parameters. The tool requires a city name, so I'll structure the JSON with "tool_name" as "get_weather" and "argument_values" containing "city": "Kanpur". That should fetch the weather data correctly.
</think>
content: {
  "tool_name": "get_weather",
  "argument_values": {
    "city": "Kanpur"
  }
}


In [20]:
import json

choosen_tool = json.loads(content)

In [28]:
if choosen_tool['tool_name'] == 'get_weather':
    weather = get_weather(choosen_tool['argument_values']["city"])
weather

35

In [30]:
# prepare the model input
user_prompt = f"you are a customer facing agent and given weather and city you can draft a respnose to user, weather = {weather} and city = {choosen_tool['argument_values']['city']}"
messages = [
    {"role": "user", "content": user_prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

model_inputs

{'input_ids': tensor([[151644,    872,    198,   9330,    525,    264,   6002,  12880,   8315,
            323,   2661,   9104,    323,   3283,    498,    646,   9960,    264,
           9039,     77,    960,    311,   1196,     11,   9104,    284,    220,
             18,     20,    323,   3283,    284,  30563,  24998, 151645,    198,
         151644,  77091,    198]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

In [31]:
# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

output_ids

[151667,
 198,
 32313,
 11,
 279,
 1196,
 6801,
 752,
 311,
 1160,
 438,
 264,
 6002,
 63306,
 8315,
 13,
 2379,
 3897,
 279,
 9104,
 438,
 220,
 18,
 20,
 323,
 279,
 3283,
 438,
 30563,
 24998,
 13,
 6771,
 752,
 1744,
 911,
 1246,
 311,
 5889,
 382,
 5338,
 11,
 358,
 1184,
 311,
 1779,
 1128,
 279,
 9104,
 374,
 13,
 220,
 18,
 20,
 374,
 264,
 9315,
 13,
 358,
 1265,
 5508,
 429,
 311,
 61347,
 476,
 68723,
 13,
 13824,
 11,
 220,
 18,
 20,
 12348,
 374,
 220,
 24,
 20,
 12348,
 68723,
 13,
 2938,
 594,
 5020,
 4017,
 13,
 2055,
 279,
 9104,
 374,
 9016,
 4017,
 382,
 7039,
 11,
 279,
 3283,
 374,
 30563,
 24998,
 11,
 892,
 374,
 304,
 6747,
 13,
 576,
 1196,
 2578,
 387,
 3330,
 369,
 9462,
 389,
 1128,
 311,
 9850,
 11,
 1128,
 311,
 653,
 11,
 476,
 7196,
 1246,
 311,
 4717,
 7010,
 13,
 8704,
 432,
 594,
 4017,
 11,
 358,
 1265,
 4190,
 3100,
 17438,
 11,
 7196,
 264,
 8896,
 11,
 323,
 1045,
 10414,
 369,
 19429,
 7010,
 382,
 40,
 1265,
 1083,
 2908,
 279,
 2266,
 13,
 576,

In [32]:
# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


thinking content: <think>
Okay, the user wants me to act as a customer-facing agent. They provided the weather as 35 and the city as Kanpur. Let me think about how to respond.

First, I need to check what the weather is. 35 is a temperature. I should convert that to Celsius or Fahrenheit. Wait, 35 degrees is 95 degrees Fahrenheit. That's pretty hot. So the weather is extremely hot.

Now, the city is Kanpur, which is in India. The user might be looking for advice on what to wear, what to do, or maybe how to stay cool. Since it's hot, I should suggest light clothing, maybe a hat, and some tips for staying cool.

I should also consider the context. The user might be a customer who needs to travel or just wants to know how to handle the weather. They might be looking for practical advice. So the response should be friendly, helpful, and provide actionable tips.

Let me structure the response. Start with a greeting, mention the weather, suggest appropriate clothing, maybe some tips for stay

In [33]:
content

'**Hi there!**  \n\nIt’s a **35°C** (95°F) day in **Kanpur**, which is extremely hot and humid! 🌡️ To stay cool, I recommend:  \n- **Wear light, breathable clothing** like cotton shirts and shorts.  \n- **Stay hydrated** with water or electrolyte drinks.  \n- **Use a fan or air conditioner** to keep the environment cool.  \n- **Avoid prolonged sun exposure** by staying indoors during peak hours (10 AM–4 PM).  \n\nLet me know if you need tips for staying comfortable or anything else! 😊'

In [35]:
import json
import re
from typing import Any, TypedDict

from langgraph.graph import END, START, StateGraph


def generate_qwen(system_prompt: str, prompt: str, *, enable_thinking: bool = False) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        pad_token_id=tokenizer.eos_token_id,
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
    return tokenizer.decode(output_ids, skip_special_tokens=True).strip()


def extract_json_object(text: str) -> dict[str, Any]:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"Model did not return JSON: {text}")
    return json.loads(match.group(0))


tool_catalog = [
    {
        "tool_name": "get_weather",
        "arguments": {"city": "string"},
        "description": "Return the weather in Celsius for a city.",
    }
]


class WeatherAgentState(TypedDict, total=False):
    user_input: str
    decision: dict[str, Any]
    observation: dict[str, Any]
    response: str


def plan_node(state: WeatherAgentState) -> WeatherAgentState:
    planner_system_prompt = (
        "You are a tool-using weather agent. "
        "Choose the correct tool when weather lookup is needed. "
        "Return only valid JSON."
    )
    planner_prompt = (
        f"Available tools: {json.dumps(tool_catalog)}\n"
        "Return either:\n"
        '1. {"tool_name": "get_weather", "arguments": {"city": "..."}}\n'
        '2. {"response_text": "...", "done": true}\n\n'
        f"User request: {state['user_input']}"
    )
    raw = generate_qwen(planner_system_prompt, planner_prompt, enable_thinking=False)
    return {"decision": extract_json_object(raw)}


def route_after_plan(state: WeatherAgentState) -> str:
    decision = state["decision"]
    if decision.get("tool_name") == "get_weather":
        return "act"
    return "respond"


def act_node(state: WeatherAgentState) -> WeatherAgentState:
    decision = state["decision"]
    city = decision["arguments"]["city"]
    weather = get_weather(city)
    return {
        "observation": {
            "tool_name": "get_weather",
            "requested_city": city,
            "result": weather,
        }
    }


def respond_node(state: WeatherAgentState) -> WeatherAgentState:
    decision = state.get("decision", {})
    if "observation" not in state:
        return {"response": decision.get("response_text", "I could not complete that request.")}

    response_system_prompt = "You are a concise customer-facing weather assistant."
    response_prompt = (
        f"User request: {state['user_input']}\n"
        f"Tool result: {json.dumps(state['observation'])}\n"
        "Write a short natural-language answer."
    )
    response = generate_qwen(response_system_prompt, response_prompt, enable_thinking=False)
    return {"response": response}


graph = StateGraph(WeatherAgentState)
graph.add_node("plan", plan_node)
graph.add_node("act", act_node)
graph.add_node("respond", respond_node)
graph.add_edge(START, "plan")
graph.add_conditional_edges("plan", route_after_plan, {"act": "act", "respond": "respond"})
graph.add_edge("act", "respond")
graph.add_edge("respond", END)

weather_agent = graph.compile()
langgraph_result = weather_agent.invoke({"user_input": user_prompt})
langgraph_result["response"]


'The current weather in Kanpur is 34°C, with a slight feeling of 35°C. The temperature is comfortable, and the wind speed is 13 km/h.'